## Installs

In [1]:
#!pip install openwakeword pyaudio
#!pip install sounddevice
#!pip install silero-vad
#!pip install faster-whisper transformers accelerate
#!pip install -U transformers huggingface_hub httpx brotli

# Imports

In [2]:
import threading
import openwakeword
import sounddevice as sd
import numpy as np 
from openwakeword.model import Model
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
import time

In [3]:
print(f"CUDA: {torch.cuda.is_available()} | Version: {torch.version.cuda} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA: True | Version: 12.1 | GPU: NVIDIA GeForce RTX 4080 Laptop GPU


In [4]:
# Downloading Predefined wakewords
from openwakeword.utils import download_models
download_models()

# Phase 1: Wake Word Listener & Thread

In [5]:
wake_word_detected = threading.Event()
def wake_word_listener():
    model = Model(wakeword_models=['hey_jarvis'], inference_framework="onnx")
    print("Listening for Wake Word...")
    with sd.InputStream(samplerate=16000, channels=1, dtype="int16", blocksize=1280) as stream:
        while not stop_event.is_set():
            if wake_word_detected.is_set():
                time.sleep(0.1)
                continue
            audio_chunk, _ = stream.read(1280)
            prediction = model.predict(audio_chunk.flatten())
            if prediction["hey_jarvis"] > 0.3:
                print(f"Wake word detected! ({prediction['hey_jarvis']:.4f})")
                wake_word_detected.set()
                time.sleep(1.5)  # cooldown to avoid re-trigger

In [13]:
wake_word_detected.clear()
t = threading.Thread(target = wake_word_listener)
t.daemon = True
t.start()

Listening for Wake Word...
Wake word detected! (0.3309)


# Phase 2: Audio Capture

## Phase 5: Transcription (Whisper Apex)

In [8]:
model_id = "Oriserve/Whisper-Hindi2Hinglish-Apex"

# device + dtype setup
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# load model
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    dtype=dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
).to(device)

# load processor (you already downloaded it, this will reuse cache)
processor = AutoProcessor.from_pretrained("openai/whisper-large-v3-turbo")
#processor = AutoProcessor.from_pretrained(model_id)

# pipeline
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=asr_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=dtype,
    device=0 if device == "cuda" else -1
)

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [9]:
def transcribe(audio):
    # audio is already float32 in [-1.0, 1.0] from sounddevice
    audio = audio.flatten().astype("float32")

    result = asr_pipeline(
        audio,
        generate_kwargs ={
            "task": "transcribe",
            "language": None,          # auto-detect each utterance, Apex handles Hinglish/English too
            "forced_decoder_ids": None # suppresses the deprecation conflict
            })
    return result["text"]

## Recording Audio part

In [10]:
vad_model, _ = torch.hub.load('snakers4/silero-vad', 'silero_vad')
SILENCE_LIMIT = 60   # ~3s at blocksize=512: 60 * 512/16000 ≈ 1.92s; tune as needed

Using cache found in C:\Users\Archit/.cache\torch\hub\snakers4_silero-vad_master


In [21]:
stop_event = threading.Event()

def record_audio():
    while not stop_event.is_set():
        wake_word_detected.wait()  # wait for "hey jarvis"
        if stop_event.is_set():
            break
        print("++++++++++++++++++++++++++++++++++++++++++++++++++|Conversation Started|++++++++++++++++++++++++++++++++++++++++++++++++++\n")
                

        # INNER loop — keeps recording until "bye jarvis"
        while not stop_event.is_set() and wake_word_detected.is_set():
            audio_buffer = []
            silence_counter = 0
            print("Recording...")
            with sd.InputStream(samplerate=16000, channels=1, dtype='float32', blocksize=512) as stream:
                while True:
                    chunk, _ = stream.read(512)
                    flat = chunk.flatten()
                    audio_buffer.append(flat)
                    ## Phase 3: Voice Activity Detection (VAD) — silero expects shape [1, 512] float32
                    
                    chunk_tensor = torch.from_numpy(flat).unsqueeze(0)  # silero needs shape [1, 512]
                    speech_prob = vad_model(chunk_tensor, 16000).item()
                    
                    # if silence detected: break     
                    if speech_prob < 0.3:
                        silence_counter += 1
                    else:
                        silence_counter = 0
                    
                    if silence_counter >= SILENCE_LIMIT:
                        break
                        
            ## Phase 4: Preprocessing
            audio = np.concatenate(audio_buffer)
            print(f"Audio length: {len(audio)} samples ({len(audio)/16000:.1f}s)")
            
            ## Phase 5: Transcription (Whisper Apex)
            transcript = transcribe(audio)
            print(f"Transcript: {transcript}")
            
            if not transcript or transcript.strip().lower() in ("", "nan"):
                if stop_event.is_set():
                    break
                print("Empty transcript, re-listening...")
                continue  # re-record, stay in conversation
                
            if any(phrase in transcript.lower() for phrase in ["bye jarvis", "bye, jarvis", "bye jarvis", "bai jarvis", "by jarvis","bye zaarves"]):
                print("\nGoodbye Have a nice day!")
                print("--------------------------------------------------|Conversation Ended|---------------------------------------------------\n")
                wake_word_detected.clear()
                break  # exit inner loop → back to waiting for wake word
                
            # Phase 6 
            output_handler(transcript)
            # no clear() — loops back to record next sentence

## Phase 6: Output Handler

In [22]:
def output_handler(transcript):
    # clean the transcript
    transcript = transcript.strip()
    transcript = transcript.lower()
    
    print(f"Output: {transcript}")
    
    # LLM hook goes here later
    return transcript

# Run STT Module

In [23]:
stop_event.clear()
wake_word_detected.clear()

t1 = threading.Thread(target=wake_word_listener, daemon=True)
t2 = threading.Thread(target=record_audio, daemon=True)
t1.start()
t2.start()

Listening for Wake Word...
Wake word detected! (0.5346)
++++++++++++++++++++++++++++++++++++++++++++++++++|Conversation Started|++++++++++++++++++++++++++++++++++++++++++++++++++

Recording...
Audio length: 64000 samples (4.0s)
Transcript: What are you doing today?
Output: what are you doing today?
Recording...
Audio length: 55296 samples (3.5s)
Transcript: Raise right hand.
Output: raise right hand.
Recording...
Audio length: 68608 samples (4.3s)
Transcript: Mujhe C210 mein le jao.
Output: mujhe c210 mein le jao.
Recording...
Audio length: 65024 samples (4.1s)
Transcript: Co circle c block.
Output: co circle c block.
Recording...
Audio length: 57344 samples (3.6s)
Transcript: Bye, Jarvis.

Goodbye Have a nice day!
--------------------------------------------------|Conversation Ended|---------------------------------------------------

Wake word detected! (0.9745)
++++++++++++++++++++++++++++++++++++++++++++++++++|Conversation Started|++++++++++++++++++++++++++++++++++++++++++++++++++


In [25]:
stop_event.set()
wake_word_detected.set()  # unblocks the .wait() so thread can exit